# Feature Engineering — Heart Disease Dataset
### مشروع تخرج — رواد مصر الرقمية

نوت بوك شامل لكل أكواد الـ Feature Engineering المستخدمة على داتا أمراض القلب (CDC BRFSS 2020).

**المحتوى:**
1. تحميل الداتا الأصلية
2. Age Numeric + Age Group
3. BMI Category + BMI Risk
4. Comorbidity Count + Level
5. Composite Risk Score + Risk Tier
6. Lifestyle Score
7. Health Score
8. Sleep Quality
9. Time Dimension (Survey Year/Quarter/Month)
10. Geographic Enrichment (US States)
11. حفظ الداتا النهائية + Star Schema

---

## 0. استدعاء المكتبات وتحميل الداتا

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)   # لضمان ثبات النتائج (reproducibility)

# غيّر المسار حسب مكان ملف الداتا عندك
DATA_PATH = "heart_2020_cleaned.xlsx"

df = pd.read_excel(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# نظرة سريعة على الـ Target Variable
print(df['HeartDisease'].value_counts())
print(f"\nنسبة الحالات الإيجابية: {(df['HeartDisease']=='Yes').mean()*100:.1f}%")

## 1. Age_Numeric + Age_Group

**الفكرة:** عمود `AgeCategory` الأصلي نصي (مثلاً "45-49") — بنحوله لرقم (نأخذ نقطة الوسط) لنقدر نستخدمه في الحسابات والـ ML، وبعدين نبسطه لمجموعات عمرية واضحة.

In [ ]:
age_map = {
    '18-24': 21, '25-29': 27, '30-34': 32, '35-39': 37,
    '40-44': 42, '45-49': 47, '50-54': 52, '55-59': 57,
    '60-64': 62, '65-69': 67, '70-74': 72, '75-79': 77,
    '80 or older': 82
}

df['Age_Numeric'] = df['AgeCategory'].map(age_map)

df['Age_Group'] = pd.cut(
    df['Age_Numeric'],
    bins=[0, 34, 54, 69, 100],
    labels=['Young (18-34)', 'Middle (35-54)', 'Senior (55-69)', 'Elderly (70+)']
).astype(str)

print(df['Age_Group'].value_counts())
df[['AgeCategory', 'Age_Numeric', 'Age_Group']].head()

## 2. BMI_Category + BMI_Risk + BMI_Risk_Score

**المصدر:** تصنيف WHO الرسمي لـ BMI.

- `BMI_Category` → 6 فئات حسب منظمة الصحة العالمية
- `BMI_Risk` → ترجمة الفئة لمستوى خطر قلبي (Low → Critical)
- `BMI_Risk_Score` → نسخة رقمية مستمرة (0–1.5) تستخدم كـ feature في الـ ML

In [ ]:
# ── BMI_Category: تصنيف WHO ──────────────────────────────────
CATEGORY_BINS = [0, 18.5, 25.0, 30.0, 35.0, 40.0, float('inf')]
CATEGORY_LABELS = [
    'Underweight', 'Normal Weight', 'Overweight',
    'Obese Class I', 'Obese Class II', 'Obese Class III'
]

df['BMI_Category'] = pd.cut(
    df['BMI'], bins=CATEGORY_BINS, labels=CATEGORY_LABELS, right=False
).astype(str)

print(df['BMI_Category'].value_counts())

In [ ]:
# ── BMI_Risk: مستوى الخطر القلبي ──────────────────────────────
RISK_MAP = {
    'Underweight'    : 'Medium',
    'Normal Weight'  : 'Low',
    'Overweight'     : 'Medium',
    'Obese Class I'  : 'High',
    'Obese Class II' : 'Very High',
    'Obese Class III': 'Critical',
}
df['BMI_Risk'] = df['BMI_Category'].map(RISK_MAP)

print(df['BMI_Risk'].value_counts())

In [ ]:
# ── BMI_Risk_Score: نسخة رقمية مستمرة (0–1.5) ──────────────────
def bmi_to_risk_score(bmi):
    if bmi < 18.5:
        return round(min((18.5 - bmi) * 0.03, 0.5), 3)
    elif bmi < 25.0:
        return 0.0
    elif bmi < 40.0:
        return round(min((bmi - 25.0) * 0.05, 1.0), 3)
    else:
        return 1.5

df['BMI_Risk_Score'] = df['BMI'].apply(bmi_to_risk_score)

# Validation: معدل أمراض القلب لازم يزيد مع كل فئة BMI
df.groupby('BMI_Category')['HeartDisease'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reindex(CATEGORY_LABELS).round(2)

## 3. Comorbidity_Count + Comorbidity_Level

**الفكرة:** عدد الأمراض المصاحبة (Stroke, Diabetes, Asthma, Kidney Disease, Skin Cancer, DiffWalking) لكل مريض — كل مرض إضافي يزيد من احتمالية أمراض القلب.

In [ ]:
# تبسيط الـ Diabetic لأن فيه 4 قيم في الأصل
df['Diabetic_Binary'] = df['Diabetic'].apply(
    lambda x: 'Yes' if 'Yes' in str(x) else 'No'
)

# الأمراض المصاحبة المعتبرة
COND_COLS = ['Stroke', 'Diabetic_Binary', 'Asthma',
             'KidneyDisease', 'SkinCancer', 'DiffWalking']

df['Comorbidity_Count'] = (df[COND_COLS] == 'Yes').sum(axis=1)

df['Comorbidity_Level'] = pd.cut(
    df['Comorbidity_Count'],
    bins=[-1, 0, 1, 2, 6],
    labels=['None', 'Low', 'Moderate', 'High']
).astype(str)

print(df['Comorbidity_Count'].value_counts().sort_index())
print()
print(df['Comorbidity_Level'].value_counts())

In [ ]:
# Validation: HD rate لازم يزيد مع عدد الأمراض المصاحبة
df.groupby('Comorbidity_Count')['HeartDisease'].apply(
    lambda x: (x == 'Yes').mean() * 100
).round(2)

## 4. Risk_Score (Composite) + Risk_Tier

**المنهجية:** Score من 0 إلى 10 بيجمع 8 عوامل خطر بأوزان مختلفة حسب أهميتها الطبية (WHO / AHA / CDC).

| العامل | الوزن | السبب |
|---|---|---|
| Smoking | 1.5 | أقوى عامل خطر قابل للتعديل |
| Diabetes | 1.5 | يضاعف خطر أمراض القلب |
| Stroke history | 1.5 | مؤشر قوي على تكرار الحالة |
| BMI | 1.5 (تدريجي) | حمل متابولي مباشر |
| Age | 1.5 (تدريجي) | عامل غير قابل للتعديل |
| Physical inactivity | 0.5 | عامل قابل للتعديل |
| Alcohol | 0.5 | يرفع ضغط الدم |
| Comorbidity count | 1.5 (max) | تراكم الأمراض |

In [ ]:
# ── حساب كل component على حدة ────────────────────────────────
smoking_score    = df['Smoking'].map({'Yes': 1.5, 'No': 0.0}).fillna(0)
diabetes_score   = df['Diabetic_Binary'].map({'Yes': 1.5, 'No': 0.0}).fillna(0)
stroke_score     = df['Stroke'].map({'Yes': 1.5, 'No': 0.0}).fillna(0)

bmi_score_map = {
    'Underweight': 0.5, 'Normal Weight': 0.0, 'Overweight': 0.5,
    'Obese Class I': 1.0, 'Obese Class II': 1.5, 'Obese Class III': 1.5
}
bmi_score = df['BMI_Category'].map(bmi_score_map).fillna(0)

def age_to_score(age):
    if   age < 40: return 0.0
    elif age < 55: return 0.5
    elif age < 70: return 1.0
    else:          return 1.5
age_score = df['Age_Numeric'].apply(age_to_score)

inactivity_score = df['PhysicalActivity'].map({'Yes': 0.0, 'No': 0.5}).fillna(0)
alcohol_score    = df['AlcoholDrinking'].map({'Yes': 0.5, 'No': 0.0}).fillna(0)
comorbidity_score = (df['Comorbidity_Count'] * 0.3).clip(upper=1.5)

In [ ]:
# ── الجمع والـ Normalization إلى 0–10 ───────────────────────
WEIGHTS = {
    'smoking': 1.5, 'diabetes': 1.5, 'stroke': 1.5, 'bmi': 1.5,
    'age': 1.5, 'inactivity': 0.5, 'alcohol': 0.5, 'comorbidity': 1.5,
}
THEORETICAL_MAX = sum(WEIGHTS.values())   # = 9.0

raw_score = (smoking_score + diabetes_score + stroke_score + bmi_score +
             age_score + inactivity_score + alcohol_score + comorbidity_score)

df['Risk_Score'] = (raw_score / THEORETICAL_MAX * 10).clip(0, 10).round(2)

# ── Risk Tiers ───────────────────────────────────────────────
df['Risk_Tier'] = pd.cut(
    df['Risk_Score'],
    bins=[0, 2.5, 5.0, 7.5, 10.0],
    labels=['Low', 'Medium', 'High', 'Critical'],
    include_lowest=True
).astype(str)

# نسخة مبسطة بـ 3 مستويات (لأن Critical نسبتها صغيرة جداً ~0.4%)
df['Risk_Tier_Simplified'] = df['Risk_Tier'].replace({'Critical': 'High'})

print(df['Risk_Tier'].value_counts())
print()
print(df['Risk_Tier_Simplified'].value_counts())

In [ ]:
# ── Validation: فرق الـ Risk Score بين المرضى والأصحاء ──────
hd_yes_mean = df[df['HeartDisease'] == 'Yes']['Risk_Score'].mean()
hd_no_mean  = df[df['HeartDisease'] == 'No']['Risk_Score'].mean()

print(f"Mean Risk Score (HeartDisease = Yes): {hd_yes_mean:.2f}")
print(f"Mean Risk Score (HeartDisease = No) : {hd_no_mean:.2f}")
print(f"Difference: +{hd_yes_mean - hd_no_mean:.2f} نقطة")

# Correlation
from scipy.stats import pointbiserialr
hd_binary = (df['HeartDisease'] == 'Yes').astype(int)
corr, pval = pointbiserialr(df['Risk_Score'], hd_binary)
print(f"\nPoint-biserial correlation: r = {corr:.4f} (p={pval:.2e})")

## 5. Lifestyle_Score + Lifestyle_Category

**الفكرة:** نقطة من 0 إلى 100 تقيس مدى صحة نمط حياة المريض، بناءً على 4 عوامل: النشاط البدني، جودة النوم، عدم التدخين، عدم الإفراط في الكحول.

In [ ]:
df['Lifestyle_Score'] = (
    (df['PhysicalActivity'] == 'Yes').astype(int) * 25 +
    df['SleepTime'].between(7, 9).astype(int) * 25 +
    (df['Smoking'] == 'No').astype(int) * 30 +
    (df['AlcoholDrinking'] == 'No').astype(int) * 20
)

df['Lifestyle_Category'] = pd.cut(
    df['Lifestyle_Score'],
    bins=[-1, 39, 59, 79, 100],
    labels=['Poor', 'Fair', 'Good', 'Excellent']
).astype(str)

print(df['Lifestyle_Category'].value_counts())

## 6. Health_Score

**الفكرة:** دمج 3 مؤشرات صحة ذاتية (GenHealth + PhysicalHealth + MentalHealth) في رقم واحد من 0 إلى 10.

In [ ]:
gen_map = {'Excellent': 5, 'Very good': 4, 'Good': 3, 'Fair': 2, 'Poor': 1}

df['Health_Score'] = (
    df['GenHealth'].map(gen_map) * 4 +
    (30 - df['PhysicalHealth']) / 30 * 3 +
    (30 - df['MentalHealth']) / 30 * 3
).round(2)

df['Health_Score'].describe()

## 7. Sleep_Quality

تصنيف ساعات النوم حسب التوصيات الطبية (7–9 ساعات للبالغين هي الأمثل).

In [ ]:
df['Sleep_Quality'] = df['SleepTime'].apply(
    lambda s: 'Optimal' if 7 <= s <= 9 else ('Short' if s < 7 else 'Excessive')
)

print(df['Sleep_Quality'].value_counts())

## 8. Survey_Year / Survey_Quarter / Survey_Month

**ملاحظة مهمة:** الداتا الأصلية من CDC BRFSS 2020 لا تحتوي على بُعد زمني تفصيلي. بنضيف توزيع زمني واقعي (محاكاة) يسمح بعمل Time Intelligence في Power BI.

In [ ]:
df['Survey_Year'] = np.random.choice(
    [2018, 2019, 2020, 2021, 2022],
    size=len(df),
    p=[0.15, 0.18, 0.22, 0.25, 0.20]
)
df['Survey_Quarter'] = np.random.randint(1, 5, size=len(df))
df['Survey_Month']   = df['Survey_Quarter'].apply(
    lambda q: np.random.randint((q-1)*3 + 1, q*3 + 1)
)

df[['Survey_Year', 'Survey_Quarter', 'Survey_Month']].head()

## 9. Geographic Enrichment — US States

**المصدر:** بيانات حقيقية من CDC BRFSS 2020 + US Census 2020 + America's Health Rankings 2020 لكل الـ 50 ولاية أمريكية.

نوزّع المرضى على الولايات بشكل واقعي حسب عدد السكان الفعلي، وبعدين نربط كل مريض بمعدل أمراض القلب الحقيقي في ولايته.

In [ ]:
# بيانات حقيقية مختصرة (نموذج — القائمة الكاملة فيها 50 ولاية)
states_data = {
    'Alabama':    [5.03, 10.8, 36.3, 20.9, 10.8, 52.0, 46, 'South'],
    'California': [39.54, 6.1, 25.1, 10.1, 7.0, 78.0, 5, 'West'],
    'Texas':      [29.73, 8.2, 33.0, 14.9, 18.4, 60.0, 34, 'South'],
    'New York':   [20.20, 6.9, 27.4, 13.7, 6.0, 72.0, 13, 'Northeast'],
    'Florida':    [21.73, 7.9, 27.4, 14.5, 11.2, 59.0, 33, 'South'],
    # ... باقي الـ 50 ولاية (راجع ملف us_states_health_data.csv للقائمة الكاملة)
}

cols = ['Population_M','HD_Prevalence_Pct','Obesity_Rate_Pct','Smoking_Rate_Pct',
        'Uninsured_Rate_Pct','Median_Income_K','Health_Rank_2020','Region']

df_states = pd.DataFrame(
    [[k] + v for k, v in states_data.items()],
    columns=['State'] + cols
)
df_states.head()

In [ ]:
# توزيع المرضى على الولايات بشكل واقعي حسب عدد السكان
pops   = df_states['Population_M'].values
states = df_states['State'].values
probs  = pops / pops.sum()

df['State'] = np.random.choice(states, size=len(df), p=probs)

# ربط كل مريض بإحصائيات ولايته
df = df.merge(
    df_states[['State', 'Region', 'HD_Prevalence_Pct', 'Health_Rank_2020']]
        .rename(columns={
            'HD_Prevalence_Pct': 'State_HD_Prevalence',
            'Health_Rank_2020': 'State_Health_Rank'
        }),
    on='State', how='left'
)

df[['PatientID', 'State', 'Region', 'State_HD_Prevalence']].head() if 'PatientID' in df.columns else df[['State','Region','State_HD_Prevalence']].head()

## 10. حفظ النتيجة النهائية

بعد كل الخطوات السابقة، الداتا أصبحت تحتوي على الـ 18 column الأصلية + أكثر من 25 column مُهندَسة (Engineered Features) جاهزة للاستخدام في:
- **Power BI** — للـ Dashboards والـ visualizations
- **SQL Server** — بعد تقسيمها لـ Star Schema (dim/fact tables)
- **Machine Learning** — كـ features مباشرة للنماذج

In [ ]:
print(f"Shape النهائي: {df.shape}")
print(f"\nكل الـ Columns:")
print(df.columns.tolist())

# حفظ النسخة الكاملة
df.to_csv('heart_disease_enriched_final.csv', index=False)
print("\n✅ تم الحفظ: heart_disease_enriched_final.csv")

---
### ملاحظة ختامية

كل الأكواد دي اتطبقت بالترتيب على نفس الداتا الأصلية، والنتيجة النهائية موجودة في ملفات:
- `heart_disease_geo_enriched.csv` (كل البيانات مجمعة)
- `heart_disease_star_schema_v2.xlsx` (مقسّمة لـ Star Schema)

الخطوة الجاية: **Machine Learning Track** — مش متضمنة في النوت بوك ده.